In [5]:
import xarray
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import scipy as sp
import pandas as pd
from netCDF4 import Dataset
import dask
import subprocess

In [16]:
#
# 
#
Expname = '' #'GammaT63L26_Warm'
DataSetname = 'vvel'
Dataname = 'v'
Outputdir = '/raid6C/kirtman/AGCM/'
Outputdir = '/Users/bmapes/Documents/AGCM_Experiments/TestT63L26_Dist/'
stamp = 'days_1-4800'
dayst = 4800
#
kmax = 26
#
#imax = 128 #### T42
#jmax = 64
imax = 192 #### T63
jmax = 96
#imax = 376 #### T124
#jmax = 188
#imax = 748 #### T248
#jmax = 374 ####
ijmax = imax*jmax

In [19]:
fps = Outputdir+Expname+'/lnps_1*.nc' # always need surface pressure
dps = xarray.open_mfdataset(fps,decode_times=True,parallel = True)
#
#
fdata = Outputdir+Expname+'/'+DataSetname+'_1*.nc'
ddata = xarray.open_mfdataset(fdata,decode_times=True,parallel = True)

Exception ignored in: <function CachingFileManager.__del__ at 0x1468e9b20>
Traceback (most recent call last):
  File "/opt/miniconda3/envs/Kirtman_agcm/lib/python3.12/site-packages/xarray/backends/file_manager.py", line 250, in __del__
    self.close(needs_lock=False)
  File "/opt/miniconda3/envs/Kirtman_agcm/lib/python3.12/site-packages/xarray/backends/file_manager.py", line 234, in close
    file.close()
  File "src/netCDF4/_netCDF4.pyx", line 2627, in netCDF4._netCDF4.Dataset.close
  File "src/netCDF4/_netCDF4.pyx", line 2590, in netCDF4._netCDF4.Dataset._close
  File "src/netCDF4/_netCDF4.pyx", line 2034, in netCDF4._netCDF4._ensure_nc_success
RuntimeError: NetCDF: Not a valid ID
Exception ignored in: <function CachingFileManager.__del__ at 0x1468e9b20>
Traceback (most recent call last):
  File "/opt/miniconda3/envs/Kirtman_agcm/lib/python3.12/site-packages/xarray/backends/file_manager.py", line 250, in __del__
    self.close(needs_lock=False)
  File "/opt/miniconda3/envs/Kirtman_a

In [ ]:
fps

In [ ]:
dps

In [ ]:
ddata

In [ ]:
#
import metpy
# Create Data Array for Control Pressure level Data geopotenial, temp, u & v
# 
#
lats = ddata['lat'].values
lons = ddata['lon'].values
plev = [1000.0,900.0,800.0,700.0,600.0,500.0,400.0,300.0,200.0,100.0,20.0]
plev_r = np.zeros(11)
for k in range(11):
    plev_r[k] = (plev[k])*100.0 # mb to Pa
#
#
tmp = (dayst,11,jmax,imax) #### "11" here corresponds to standard pressure levels not to model levels
dout = np.zeros(tmp)
pressure = np.zeros((kmax,jmax,imax))
siglevs = ddata['lev']
for k in range (dayst):
    vv = ddata[Dataname][k,:,:,:]
    ps = dps.lnps[k,:,:]
    surfp = (np.exp(ps))*1000.0*100.0 # in Pa
    for kk in range(kmax):
        pressure[kk,:,:] = surfp[:,:]*siglevs[kk]
    vv = vv.compute()
    ps = ps.compute()
    dout[k] = metpy.interpolate.log_interpolate_1d(plev_r,pressure,vv, axis=0)
#
times = ddata['time']
dData = xarray.Dataset({Dataname: (['time','lev','lat','lon'],dout)},
                        coords={'time': times,'lev':plev, 'lat': lats, 'lon': lons})

In [ ]:
dData

In [ ]:
dData.to_netcdf(Outputdir+Expname+'/'+DataSetname+'_Pressure_'+stamp+'.nc')

In [ ]:
dData[Dataname][400,0,:,:].plot()

In [ ]:
dData[Dataname][100,7,:,:].plot()

In [ ]:
dData[Dataname][100,:,45,64].plot()